# 04 — Inheritance

Inheritance is another pillar of Object-Oriented Programming. It lets one class (the **child** / **subclass**) reuse and extend the attributes and methods of another class (the **parent** / **superclass**).

In this notebook we will cover:

1. Why inheritance matters
2. Basic single inheritance
3. The `super()` function
4. Overriding methods
5. Multi-level inheritance
6. Multiple inheritance and the MRO (Method Resolution Order)
7. `isinstance()` vs `issubclass()`
8. Abstract base classes
9. A complete real-world example: a shape hierarchy
10. Practice exercises


## 1. Why Inheritance Matters

Without inheritance, you'd copy-paste the same code across many similar classes. Inheritance lets you:

- **Reuse code** already written in a parent class
- **Extend** behavior by adding new attributes/methods in the child
- **Override** behavior by redefining a parent's method in the child
- Model natural **"is-a" relationships** (a `Dog` *is an* `Animal`, a `Car` *is a* `Vehicle`)

Let's see the problem first, without inheritance.

In [ ]:
class Dog:
    def __init__(self, name):
        self.name = name
    def eat(self):
        print(f"{self.name} is eating")
    def speak(self):
        print(f"{self.name} says Woof!")

class Cat:
    def __init__(self, name):
        self.name = name
    def eat(self):
        print(f"{self.name} is eating")   # duplicated logic
    def speak(self):
        print(f"{self.name} says Meow!")

d = Dog("Rex")
c = Cat("Whiskers")
d.eat(); d.speak()
c.eat(); c.speak()


`eat()` is duplicated in both classes. Inheritance lets us factor that shared behavior into a common parent class.

## 2. Basic Single Inheritance

To inherit from a class, put the parent class name in parentheses after the child class name: `class Child(Parent):`.

The child automatically gets all public and protected attributes/methods of the parent.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def eat(self):
        print(f"{self.name} is eating")

    def speak(self):
        print(f"{self.name} makes a sound")


class Dog(Animal):        # Dog inherits from Animal
    pass                   # no new behavior yet


class Cat(Animal):
    pass


d = Dog("Rex")
c = Cat("Whiskers")

d.eat()     # inherited from Animal
c.eat()     # inherited from Animal
d.speak()   # inherited generic behavior


## 3. The `super()` Function & 4. Overriding Methods

Usually a child class wants to **override** a method to give it specialized behavior, while still reusing the parent's implementation where useful. `super()` gives you access to the parent class's methods from within the child.

- **Override**: redefine a method with the same name in the child class.
- **`super().__init__(...)`**: call the parent's constructor so it can set up shared attributes, before adding child-specific ones.
- **`super().method(...)`**: call the parent's version of a method, then add extra behavior.

In [ ]:
class Animal:
    def __init__(self, name, sound="..."):
        self.name = name
        self.sound = sound

    def eat(self):
        print(f"{self.name} is eating")

    def speak(self):
        print(f"{self.name} says {self.sound}")


class Dog(Animal):
    def __init__(self, name):
        super().__init__(name, sound="Woof!")   # reuse parent's constructor
        self.tricks = []

    def speak(self):                             # override
        super().speak()                          # still use parent's logic...
        print(f"{self.name} wags its tail")       # ...then add more

    def learn_trick(self, trick):
        self.tricks.append(trick)


class Cat(Animal):
    def __init__(self, name):
        super().__init__(name, sound="Meow!")

    def speak(self):                              # full override, no super() call
        print(f"{self.name} says {self.sound} (and ignores you)")


d = Dog("Rex")
d.speak()
d.learn_trick("sit")
print(d.tricks)

c = Cat("Whiskers")
c.speak()


## 5. Multi-Level Inheritance

A class can inherit from a class that itself inherits from another class, forming a chain: `GrandParent -> Parent -> Child`. Each level can add or override behavior, and `super()` always refers to the *next* class up the chain (based on the MRO, see below).

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def describe(self):
        print(f"{self.name} is an animal")


class Mammal(Animal):
    def __init__(self, name, fur_color):
        super().__init__(name)
        self.fur_color = fur_color

    def describe(self):
        super().describe()
        print(f"{self.name} is a mammal with {self.fur_color} fur")


class Dog(Mammal):
    def __init__(self, name, fur_color, breed):
        super().__init__(name, fur_color)
        self.breed = breed

    def describe(self):
        super().describe()
        print(f"{self.name} is a {self.breed} dog")


rex = Dog("Rex", "brown", "Labrador")
rex.describe()


## 6. Multiple Inheritance & the MRO

Python allows a class to inherit from **more than one** parent class: `class Child(ParentA, ParentB):`.

When several parents define the same method, Python decides which one to use based on the **Method Resolution Order (MRO)** — computed with the **C3 linearization** algorithm. You can inspect it with `ClassName.__mro__` or `ClassName.mro()`.

In [ ]:
class Flyer:
    def move(self):
        print("Flying through the air")

class Swimmer:
    def move(self):
        print("Swimming through water")

class Duck(Flyer, Swimmer):   # Flyer listed first -> takes priority
    pass

d = Duck()
d.move()                       # uses Flyer.move because Flyer comes first

print(Duck.__mro__)


If `Duck` needs *both* behaviors, it can override `move()` and call each parent's version explicitly:

In [ ]:
class Duck(Flyer, Swimmer):
    def move(self):
        Flyer.move(self)
        Swimmer.move(self)

Duck().move()


## 7. `isinstance()` vs `issubclass()`

- **`isinstance(obj, Class)`** — checks if `obj` is an instance of `Class` (or any of its subclasses).
- **`issubclass(ChildClass, ParentClass)`** — checks if `ChildClass` inherits from `ParentClass`.

In [ ]:
class Animal: pass
class Dog(Animal): pass
class Cat(Animal): pass

rex = Dog()

print(isinstance(rex, Dog))      # True: direct instance
print(isinstance(rex, Animal))   # True: Dog IS-A Animal
print(isinstance(rex, Cat))      # False

print(issubclass(Dog, Animal))   # True
print(issubclass(Animal, Dog))   # False
print(issubclass(Dog, Dog))      # True (a class is a subclass of itself)


## 8. Abstract Base Classes

Sometimes a parent class should never be instantiated directly — it only exists to define a **common interface** that subclasses must implement. Python's `abc` module supports this with `ABC` and `@abstractmethod`.

If a subclass doesn't implement every abstract method, Python raises a `TypeError` when you try to instantiate it.

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

    @abstractmethod
    def perimeter(self):
        ...

    def describe(self):
        print(f"Area: {self.area():.2f}, Perimeter: {self.perimeter():.2f}")


try:
    s = Shape()          # cannot instantiate an abstract class
except TypeError as e:
    print("TypeError:", e)


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


r = Rectangle(4, 5)
r.describe()


## 9. Complete Example: A Shape Hierarchy

Let's combine everything into a small hierarchy: an abstract `Shape` base class, with `Circle`, `Rectangle`, and `Square` (which itself inherits from `Rectangle`, demonstrating multi-level inheritance).

In [ ]:
from abc import ABC, abstractmethod
import math


class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

    @abstractmethod
    def perimeter(self):
        ...

    def __repr__(self):
        return f"{self.__class__.__name__}(area={self.area():.2f}, perimeter={self.perimeter():.2f})"


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    def perimeter(self):
        return 2 * (self.width + self.height)


class Square(Rectangle):               # multi-level: Square -> Rectangle -> Shape
    def __init__(self, side):
        super().__init__(side, side)   # a square is a rectangle with equal sides


shapes = [Circle(3), Rectangle(4, 5), Square(4)]

for shape in shapes:
    print(shape)
    print("  is a Shape:", isinstance(shape, Shape))

total_area = sum(s.area() for s in shapes)
print(f"Total area: {total_area:.2f}")


Notice how every shape can be treated uniformly through the common `Shape` interface (`area()`, `perimeter()`), even though each computes them differently. This is inheritance working together with **polymorphism** — the topic of the next notebook.

## 10. Practice Exercises

Try these on your own in the empty cell(s) below.

1. Create a base class `Vehicle` with `make`, `model`, and a method `info()`. Create `Car` and `Motorcycle` subclasses that override `info()` to add vehicle-specific details, using `super()`.
2. Create an abstract class `Employee` with an abstract method `calculate_pay()`. Implement `SalariedEmployee` and `HourlyEmployee` subclasses with different pay calculations.
3. Create three classes `A`, `B`, `C` where `C(A, B)` and both `A` and `B` define a method `greet()`. Print `C.__mro__` and explain (in a markdown cell) which `greet()` gets called and why.


In [ ]:
# Exercise 1: Vehicle hierarchy (write your solution here)




In [ ]:
# Exercise 2: Employee pay hierarchy (write your solution here)




In [ ]:
# Exercise 3: MRO exploration (write your solution here)




## Summary

- **Inheritance** lets a child class reuse and extend a parent class's attributes and methods, modeling "is-a" relationships.
- **`super()`** gives access to the parent class's methods, most commonly inside `__init__` and when overriding a method.
- **Overriding** lets a subclass replace or extend a parent's method.
- Classes can chain through **multi-level inheritance**, and Python also supports **multiple inheritance**, resolved via the **MRO** (`ClassName.__mro__`).
- **`isinstance()`** checks object-to-class relationships (including inherited ones); **`issubclass()`** checks class-to-class relationships.
- **Abstract base classes** (`ABC` + `@abstractmethod`) let you define a required interface that subclasses must implement, and prevent the base class itself from being instantiated.

**Next up:** `05_Polymorphism.ipynb`
